# Machine Learning for Applied Research

## 9: Machine Learning Pipelines

In this notebook, we will cover:
- Why preprocessing needs to live inside the model, not before it
- `ColumnTransformer`: applying different preprocessing to different columns
- Scaling and its effect on linear model coefficients
- Tuning hyperparameters inside a full pipeline
- Feature selection as an optional, explicit trade-off

### Introduction

Throughout this course we have done preprocessing somewhat by hand, a quick ordinal mapping for AI4I's `Type` column in Notebook 6, a `StandardScaler` fit separately in Notebook 7. Today formalizes this into a single, reproducible object: a scikit-learn `Pipeline`. This matters for two reasons that connect directly to earlier notebooks: it prevents the kind of data leakage Notebook 7 demonstrated, since a pipeline refits every preprocessing step inside each cross-validation fold rather than on the full dataset, and it replaces Notebook 6's quick-but-imperfect categorical encoding with something done properly.

### Why Preprocessing Belongs Inside the Model, Not Before It

Doing preprocessing "by hand" (fit a scaler, transform the data, then split, or split first but still manage every step yourself across every cross-validation fold) is error-prone, especially once hyperparameter search is involved. A `Pipeline` bundles preprocessing steps and a model together into a single object that still supports `.fit()` and `.predict()` like any other estimator, but internally guarantees that every preprocessing step is fit only on the current training fold, never on data it shouldn't see.

### Meet ColumnTransformer: Different Preprocessing for Different Columns

AI4I's features mix a categorical column (`Type`: L, M, or H) with numeric sensor readings on very different scales (temperature around 300 K, rotational speed around 1,500 rpm, torque around 40 Nm). A `ColumnTransformer` applies a different transformation to different columns in one step: `OneHotEncoder` for the categorical column, `StandardScaler` for the numeric ones.

In [ ]:
!pip install ucimlrepo --quiet

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

maintenance = fetch_ucirepo(id=601)
feature_cols = ["Type", "Air temperature", "Process temperature",
                 "Rotational speed", "Torque", "Tool wear"]
maint_df = pd.concat(
    [maintenance.data.features[feature_cols], maintenance.data.targets[["Machine failure"]]],
    axis=1
)

X_maint = maint_df.drop(columns=["Machine failure"])
y_maint = maint_df["Machine failure"]

X_maint_train, X_maint_test, y_maint_train, y_maint_test = train_test_split(
    X_maint, y_maint, test_size=0.25, random_state=42, stratify=y_maint
)

print(X_maint_train.dtypes)

Notice `Type` is left as-is this time! Text categories, no manual mapping like Notebook 6. We will let the pipeline handle it properly below.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

categorical_features = ["Type"]
numeric_features = ["Air temperature", "Process temperature",
                     "Rotational speed", "Torque", "Tool wear"]

preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("num", StandardScaler(), numeric_features),
])

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42)),
])

pipeline.fit(X_maint_train, y_maint_train)
print(f"Pipeline test accuracy: {pipeline.score(X_maint_test, y_maint_test):.3f}")

`OneHotEncoder(handle_unknown="ignore")` is a small but important detail: it tells the encoder what to do if it ever sees a category at prediction time that it did not see during training, rather than raising an error. When you call `pipeline.fit(...)`, the preprocessor is fit on the training data only; when you call `pipeline.score(...)` or `pipeline.predict(...)` on the test set, the preprocessor only *transforms* using what it already learned from training! It never refits! This is the leakage-safe procedure from Notebook 7, now automated instead of something you have to remember to do correctly by hand.

#### 1.1 Exercises:
> (a) Run 5-fold `StratifiedKFold` cross-validation (`scoring="roc_auc"`) on the entire `pipeline` object - not just the classifier - using `X_maint_train`, `y_maint_train`.
>
> (b) Print the mean and standard deviation of the CV ROC AUC.
>
> (c) Compare this to Notebook 7's Challenge result, which used a manual ordinal mapping (L=0, M=1, H=2) for `Type` instead of one-hot encoding. Did proper encoding help, hurt, or make little difference? Can you think of a reason a tree-based model in particular might not care much either way? (Hint: think about what a single decision-tree split actually does with an ordinally-coded categorical feature.)

In [ ]:
# Your solution:





##### Solution:


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipeline_cv_scores = cross_val_score(pipeline, X_maint_train, y_maint_train, cv=skf, scoring="roc_auc")

print(f"Pipeline CV ROC AUC per fold: {pipeline_cv_scores}")
print(f"Mean: {pipeline_cv_scores.mean():.3f} (+/- {pipeline_cv_scores.std():.3f})")

# (c) For a tree-based model like Random Forest, proper one-hot encoding
# often makes little practical difference compared to an arbitrary
# ordinal mapping: a tree can still find a useful split point on an
# ordinally-coded feature (e.g. "is Type <= 1?"), even though the numeric
# order does not reflect a genuine ranking. The difference matters far
# more for linear models, where an ordinal encoding implicitly claims
# that the gap between L and M means the same thing as the gap between M
# and H, which is an assumption that is often simply wrong.

#### 1.2 Exercises:
> (a) Call `pipeline.named_steps["preprocessor"].get_feature_names_out()` and print the result.
>
> (b) How many columns came out, compared to the 6 columns that went in? Where did the extra ones come from?
>
> (c) Match each output name back to the original column it came from (the `cat__` and `num__` prefixes are a hint).

In [ ]:
# Your solution:





##### Solution:


In [ ]:
encoded_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
print(encoded_names)
print(f"\n{len(encoded_names)} columns out, from 6 columns in")

# (c) The num__ prefixed names (num__Air temperature, etc.) are the
# 5 numeric columns, scaled but otherwise unchanged one-to-one. The
# cat__ prefixed names are the one-hot encoding of Type: cat__Type_H,
# cat__Type_L, cat__Type_M - one binary column per category, which is
# why Type alone accounts for 3 of the output columns instead of 1.

### Scaling and Its Effect on Linear Model Coefficients

Notebook 4 flagged something we need to resolve: raw-feature linear regression coefficients are not directly comparable, because features are measured in wildly different units and ranges. Scaling fixes this.

In [ ]:
from sklearn.linear_model import LinearRegression

energy = fetch_ucirepo(id=242)
X_e = energy.data.features
y_e = energy.data.targets

X_e.columns = ["Relative Compactness",
             "Surface Area",
             "Wall Area",
             "Roof Area",
             "Overall Height",
             "Orientation",
             "Glazing Area",
             "Glazing Area Distribution"]

y_e.columns = ["Heating Load", "Cooling Load"]

target_col_e = y_e.columns[0]
df_e = pd.concat([X_e, y_e[[target_col_e]]], axis=1)

X_e_train, X_e_test, y_e_train, y_e_test = train_test_split(
    df_e.drop(columns=[target_col_e]), df_e[target_col_e], test_size=0.25, random_state=42
)

scaled_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression()),
])
scaled_pipeline.fit(X_e_train, y_e_train)

scaled_coefficients = pd.Series(
    scaled_pipeline.named_steps["regressor"].coef_, index=X_e_train.columns
).sort_values(key=abs, ascending=False)

print("Coefficients after scaling (comparable magnitudes now):")
print(scaled_coefficients)

Compare this ranking to Notebook 4's raw-feature coefficients. The order may well have changed, because every coefficient is now expressed in "per standard deviation" terms rather than "per raw unit", which is what makes them fairly comparable to each other. Note that scaling changes nothing about a Random Forest or decision tree's predictions - trees split on the order of values, not their scale, which is why we did not need to worry about this in Notebook 4's tree-based models.

#### 1.3 Exercises:
> Scaling doesn't just affect interpretation: for a regularized model like Ridge (Notebook 10), it affects actual predictions, since the penalty is applied to the coefficients directly, and a feature on a small scale needs a proportionally larger coefficient to have the same effect.
>
> (a) Build two pipelines: one with `Ridge(alpha=10.0)` fit directly on unscaled `X_e_train`, and one with `StandardScaler()` followed by `Ridge(alpha=10.0)`.
>
> (b) Compare their cross-validated RMSE (`cv=5`, `scoring="neg_root_mean_squared_error"`) on `X_e_train`/`y_e_train`.
>
> (c) Does scaling before Ridge help, hurt, or make little difference here? Why would you generally expect scaling to matter more for Ridge than for plain `LinearRegression`?

In [ ]:
# Your solution:




##### Solution:


In [ ]:
from sklearn.linear_model import Ridge

ridge_unscaled = Ridge(alpha=10.0)
ridge_scaled = Pipeline(steps=[("scaler", StandardScaler()), ("regressor", Ridge(alpha=10.0))])

unscaled_scores = cross_val_score(ridge_unscaled, X_e_train, y_e_train, cv=5, scoring="neg_root_mean_squared_error")
scaled_scores = cross_val_score(ridge_scaled, X_e_train, y_e_train, cv=5, scoring="neg_root_mean_squared_error")

print(f"Ridge, unscaled - Mean CV RMSE: {-unscaled_scores.mean():.2f}")
print(f"Ridge, scaled   - Mean CV RMSE: {-scaled_scores.mean():.2f}")

# (c) Scaling typically helps Ridge noticeably here. Ridge's penalty
# shrinks every coefficient by roughly the same amount regardless of
# that feature's scale, so an unscaled small-range feature (like
# Relative Compactness) gets penalized just as harshly as a large-range
# one (like Surface Area), even though it needs a much bigger
# coefficient to have a comparable effect on the prediction. Plain
# LinearRegression has no penalty term, so this issue never arises for
# it: only the interpretation of its coefficients was affected, not its
# predictions.

### Tuning Inside a Pipeline

Grid and random search work on a `Pipeline` exactly as they do on a plain model. You just need a way to tell the search which step a hyperparameter belongs to. scikit-learn uses a `step_name__parameter_name` naming convention for this.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_pipeline = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [4, 6, 8],
}

grid_search_pipeline = GridSearchCV(
    pipeline, param_grid=param_grid_pipeline, cv=skf, scoring="roc_auc", n_jobs=-1
)
grid_search_pipeline.fit(X_maint_train, y_maint_train)

print(f"Best parameters: {grid_search_pipeline.best_params_}")
print(f"Best CV ROC AUC: {grid_search_pipeline.best_score_:.3f}")

#### 1.4 Exercises:
> (a) Import `roc_auc_score` from `sklearn.metrics`, and compute the test ROC AUC of `grid_search_pipeline.best_estimator_` on `X_maint_test`/`y_maint_test` (use `.predict_proba(...)[:, 1]`).
>
> (b) Compute the test ROC AUC of the original, untuned `pipeline` the same way, and compare the two.
>
> (c) Because tuning happened entirely inside cross-validation on `X_maint_train`, with the preprocessing refit inside every fold too, is this comparison safe from the leakage issues discussed in Notebook 7? Explain briefly.

In [ ]:
# Your solution:





##### Solution:


In [ ]:
from sklearn.metrics import roc_auc_score

tuned_pipeline_auc = roc_auc_score(
    y_maint_test, grid_search_pipeline.best_estimator_.predict_proba(X_maint_test)[:, 1]
)
untuned_pipeline_auc = roc_auc_score(
    y_maint_test, pipeline.predict_proba(X_maint_test)[:, 1]
)

print(f"Tuned pipeline   - Test ROC AUC: {tuned_pipeline_auc:.3f}")
print(f"Untuned pipeline - Test ROC AUC: {untuned_pipeline_auc:.3f}")

# (c) Yes. Because the OneHotEncoder and StandardScaler live inside the
# Pipeline, GridSearchCV automatically refits them fresh inside every
# training fold during the search, and X_maint_test was never touched
# until this final evaluation. This is the leakage-safe workflow
# Notebook 7 described by hand. A Pipeline just makes it structurally
# impossible to get wrong by accident.

#### 1.5 Exercises (discussion):
> (a) Sketch out (in words, or as a short list of step names) what the preprocessing pipeline for your own research data would look like: what needs scaling, what needs encoding, what (if anything) needs imputing for missing values?
>
> (b) Is there a step in your sketch that you have been doing "by hand" so far, the way this course did before today? What would change if you wrapped it in a `Pipeline`?

In [ ]:
# Your solution:





##### Solution:

In [ ]:
# Discussion exercise, no fixed answer. A good sign your sketch is
# concrete enough: could you write the actual ColumnTransformer for it
# right now, listing which named columns go into which transformer? If
# any step is vague ("clean up the data somehow"), that is worth
# sharpening before you sit down to code it.

#### Challenge:
> Build a new pipeline that adds a feature-selection step after preprocessing: use `SelectFromModel(RandomForestClassifier(random_state=42), max_features=3)` to keep only the 3 most useful features (after encoding), then fit a `LogisticRegression(max_iter=1000)` on the reduced set. How does its cross-validated ROC AUC compare to the full pipeline above? Would you use this reduced model in practice: what would you gain, and what might you risk losing?

In [ ]:
# Your solution:





##### Solution:


In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression

selective_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("selector", SelectFromModel(RandomForestClassifier(random_state=42), max_features=3)),
    ("classifier", LogisticRegression(max_iter=1000)),
])

selective_scores = cross_val_score(selective_pipeline, X_maint_train, y_maint_train, cv=skf, scoring="roc_auc")

print(f"Reduced-feature pipeline - Mean CV ROC AUC: {selective_scores.mean():.3f} (+/- {selective_scores.std():.3f})")
print(f"Full pipeline (for comparison) - Mean CV ROC AUC: {pipeline_cv_scores.mean():.3f}")

# Cutting down to just 3 features usually costs at least a little
# accuracy compared to using everything, though sometimes barely any. The
# potential gain: a simpler, faster, easier-to-explain model.
# This is valuable if you need to report which few sensors matter most to
# a non-technical audience. The risk: a feature that looks unimportant on
# this training data might still matter for cases the model has not seen,
# and discarding it removes any chance of the model using it.

## Conclusion

This notebook turned informal, error-prone preprocessing into something structural:

- **`ColumnTransformer`**: different preprocessing for different column types, in one object, with `get_feature_names_out()` to see exactly what came out the other side
- **`Pipeline`**: bundles preprocessing and a model together, and automatically prevents the leakage pattern from Notebook 7
- **Scaling**: essential for making linear model coefficients comparable, and for Ridge specifically, essential for correct predictions too (not just interpretation)
- **Tuning inside a pipeline**: the `step__parameter` naming convention lets `GridSearchCV` and `RandomizedSearchCV` tune preprocessing and model hyperparameters together
- **Feature selection**: an explicit, visible trade-off between simplicity/interpretability and using all available information